In [1]:
import pandas as pd
import re
import string
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

In [2]:
file_path = '/kaggle/input/review/reviews.csv'

In [3]:
df = pd.read_csv(file_path)
df['rating'] = pd.to_numeric(df['rating'], errors='coerce').astype('Int64')
df.dropna(subset=['text', 'rating'], inplace=True)
df['rating'] = df['rating'].astype(int)
print(f"Successfully loaded {len(df)} rows.")

Successfully loaded 946304 rows.


In [4]:
print(df[['text', 'rating']].head())

                                                text  rating
0  So I ordered a tampiqueña thinking i was going...       1
1  Very bad service, the girl who helped us was n...       1
2  Food was good.  I ordered the flautas.  They w...       4
3  Food was great.  I had beef tacos my wife had ...       4
4  Asked for a chicken fried steak. Got grilled c...       1


In [5]:
!pip install nltk

In [6]:
import pandas as pd
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

In [8]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Error loading punkt: <urlopen error [Errno -3] Temporary
[nltk_data]     failure in name resolution>
[nltk_data] Error loading stopwords: <urlopen error [Errno -3]
[nltk_data]     Temporary failure in name resolution>


KeyboardInterrupt: 

In [10]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

In [11]:
def advanced_clean(text):
    text = re.sub(r'[^a-zA-Z\s]', '', text.lower())
    tokens = word_tokenize(text)
    cleaned_tokens = [
        lemmatizer.lemmatize(token) 
        for token in tokens 
        if token not in stop_words and len(token) > 2
    ]
    return " ".join(cleaned_tokens)

df['cleaned_text'] = df['text'].apply(advanced_clean)

In [12]:
df.to_csv('/kaggle/working/reviews_cleaned.csv')

In [13]:
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['rating'])

In [14]:
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['rating'])

In [15]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('nb', MultinomialNB())
])

parameters = {
    'tfidf__max_df': (0.5, 0.75, 1.0),
    'tfidf__ngram_range': ((1, 1), (1, 2)),  # Unigrams or Bigrams
    'nb__alpha': (0.1, 0.5, 1.0)              
}

In [16]:
grid_search = GridSearchCV(pipeline, parameters, cv=5, n_jobs=-1, verbose=1)
grid_search.fit(train_df['cleaned_text'], train_df['rating'])

Fitting 5 folds for each of 18 candidates, totalling 90 fits


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('tfidf', TfidfVectorizer()),
                                       ('nb', MultinomialNB())]),
             n_jobs=-1,
             param_grid={'nb__alpha': (0.1, 0.5, 1.0),
                         'tfidf__max_df': (0.5, 0.75, 1.0),
                         'tfidf__ngram_range': ((1, 1), (1, 2))},
             verbose=1)

In [17]:
best_clf = grid_search.best_estimator_
best_params = grid_search.best_params_
best_score = grid_search.best_score_

In [18]:
print(f"Best Score: {grid_search.best_score_}")
print(f"Best Params: {grid_search.best_params_}")

Best Score: 0.7008294104606954
Best Params: {'nb__alpha': 0.1, 'tfidf__max_df': 0.5, 'tfidf__ngram_range': (1, 2)}


In [20]:
best_clf = pickle.load(open(filename, 'rb'))
y_pred = best_clf.predict(test_df['cleaned_text'])

In [21]:
unique_ratings = sorted(df['rating'].unique())
target_names_5class = [f'Rating {r}' for r in unique_ratings]

In [22]:
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
accuracy = accuracy_score(test_df['rating'], y_pred)
report = classification_report(test_df['rating'], y_pred, labels=unique_ratings, target_names=target_names_5class, zero_division=0)
conf_mat = confusion_matrix(test_df['rating'], y_pred)

In [23]:
print(f"Accuracy: {accuracy:.4f}\n")
print("Classification Report:\n", report)
print("\nConfusion Matrix:\n", conf_mat)

Accuracy: 0.7018

Classification Report:
               precision    recall  f1-score   support

    Rating 1       0.61      0.73      0.67      7170
    Rating 2       0.40      0.01      0.01      3906
    Rating 3       0.42      0.08      0.14      6923
    Rating 4       0.40      0.17      0.24     17239
    Rating 5       0.74      0.97      0.84     59393

    accuracy                           0.70     94631
   macro avg       0.52      0.39      0.38     94631
weighted avg       0.63      0.70      0.63     94631


Confusion Matrix:
 [[ 5241    20   231   347  1331]
 [ 1703    29   352   630  1192]
 [ 1038    17   566  1911  3391]
 [  315     4   160  2924 13836]
 [  232     2    42  1468 57649]]


In [24]:
new_reviews = [
    "This is absolutely terrible, the worst I have seen.",
    "The purchase was great! I'm completely satisfied.", 
    "It works fine, I guess. Nothing exciting.", 
    "A good choice, worth the money.", 
    "Below average, slightly disappointed.", 
]

predictions = best_clf.predict(new_reviews)
for review, pred in zip(new_reviews, predictions):
    print(f"Review: '{review}'\nPredicted Rating: {pred} / 5\n")

Review: 'This is absolutely terrible, the worst I have seen.'
Predicted Rating: 1 / 5

Review: 'The purchase was great! I'm completely satisfied.'
Predicted Rating: 5 / 5

Review: 'It works fine, I guess. Nothing exciting.'
Predicted Rating: 3 / 5

Review: 'A good choice, worth the money.'
Predicted Rating: 5 / 5

Review: 'Below average, slightly disappointed.'
Predicted Rating: 3 / 5



In [19]:
import pickle

filename = 'nb_model.sav'
pickle.dump(best_clf, open(filename, 'wb'))